In [1]:
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [2]:
import keras_nlp
from keras_nlp import models
print(keras_nlp.__version__)

0.8.2


In [3]:
! pip install tensorflow_datasets

  Using cached click-8.1.7-py3-none-any.whl.metadata (3.0 kB)
  Preparing metadata (setup.py) ... done
  Using cached fsspec-2024.2.0-py3-none-any.whl.metadata (6.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 396.8 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 810.8 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.4/152.4 kB 515.3 kB/s eta 0:00:00a 0:00:01
Using cached click-8.1.7-py3-none-any.whl (97 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.5/126.5 kB 641.7 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 723.4 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.7/228.7 kB 922.8 kB/s eta 0:00:00a 0:00:01
Using cached fsspec-2024.2.0-py3-none-any.whl (170 kB)
  Created wheel for promise: filename=promise-2.3-py3-none-any.whl size=21484 sha256=8d08e4995ea126cd0f566a2c1a1a7b8bdd8161d61f26e628463628b00d24886c
  Stored in directory: /home/hueyan/.cache

In [4]:
import keras
print(keras.__version__)

3.0.5


In [5]:
import numpy as np
import keras_nlp
import tensorflow_datasets as tfds

imdb_train, imdb_test = tfds.load(
    "imdb_reviews",
    split=["train", "test"],
    as_supervised=True,
    batch_size=16,
)
model_name = "bert_tiny_en_uncased_sst2"

preprocessor = keras_nlp.models.BertPreprocessor.from_preset(
    model_name,
    sequence_length=128,
)

/home/hueyan/anaconda3/envs/keras-nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dl Completed...: 0 url [00:00, ? url/s]
Generating train examples...:  95%|████████████████████████████████████▎ | 23866/25000 [00:04<00:00, 6234.49 examples/s]
                                                                                                                        
Shuffling /home/hueyan/tensorflow_datasets/imdb_reviews/plain_text/1.0.0.incomplete5FKVRQ/imdb_reviews-train.tfrecord*..
Generating test examples...:  78%|██████████████████████████████▎        | 19447/25000 [00:03<00:00, 6747.94 examples/s]
                                                                                                                        
Shuffling /home/hueyan/tensorflow_datasets/imdb_reviews/plain_text/1.0.0.incomplete5FKVRQ/imdb_reviews-test.tfrecord*...
Generating unsupervised examples...:  94%|█████████████████████████████▏ | 47110/50000 [00:09<00:00, 6951.17 examples/s]
                                                                                                                 

Dataset imdb_reviews downloaded and prepared to /home/hueyan/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


100%|██████████████████████████████████████████████████████████████████████████████████| 547/547 [00:00<00:00, 2.34MB/s]
100%|█████████████████████████████████████████████████████████████████████████████████| 226k/226k [00:01<00:00, 231kB/s]


In [6]:
classifier = keras_nlp.models.BertClassifier.from_preset(
    model_name,
    num_classes=2,
    activation="sigmoid",
    preprocessor=preprocessor
)
classifier.fit(imdb_train, validation_data=imdb_test)

class_names = ["negative", "postive"]
review1 = "What an amazing movie!"
scores = classifier.predict([review1])
print(review1)
print(class_names[np.argmax(scores)], "-", scores)
review2 = "A total waste of my time."
scores = classifier.predict([review2])
print(review2)
print(class_names[np.argmax(scores)], "-", scores)

100%|██████████████████████████████████████████████████████████████████████████████| 2.14k/2.14k [00:00<00:00, 3.50MB/s]
100%|███████████████████████████████████████████████████████████████████████████████| 16.8M/16.8M [00:28<00:00, 627kB/s]


   3/1563 ━━━━━━━━━━━━━━━━━━━━ 52s 34ms/step - loss: 0.4983 - sparse_categorical_accuracy: 0.8472   

I0000 00:00:1709514874.692093     793 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
W0000 00:00:1709514874.777447     793 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


1563/1563 ━━━━━━━━━━━━━━━━━━━━ 57s 25ms/step - loss: 0.4735 - sparse_categorical_accuracy: 0.7720 - val_loss: 0.4107 - val_sparse_categorical_accuracy: 0.8078
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
What an amazing movie!
postive - [[0.07241403 0.9293244 ]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 263ms/step
A total waste of my time.
negative - [[0.9023546  0.09972662]]
